# 03 · Modelos clásicos: Naive Bayes y SVM (Fase III)

Resultados en **validación** de Complement NB y LinearSVC calibrado, cada uno en versión
**jerárquica** (1 modelo de dominio + 1 de área por dominio) y **plana** (las 136 áreas), con
TF-IDF (secciones 6.3 a 6.6 del diseño).

El entrenamiento no se hace aquí; se ejecuta antes desde la terminal:

```bash
python -m clasificador_ml.entrenamiento --config ml/configs/naive_bayes.yaml
python -m clasificador_ml.entrenamiento --config ml/configs/svm.yaml
```

Este notebook lee `models/<algoritmo>/<version>/` (la versión más reciente si no se indica).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from clasificador_ml.artefactos import cargar_metadatos, cargar_modelo, cargar_predicciones
from clasificador_ml.evaluacion import matriz_confusion, reporte_por_clase
from clasificador_ml.rutas import DATOS_PROCESADOS, MODELOS

# Versión de cada algoritmo (None = la más reciente). SICDA_MODELOS cambia la carpeta.
DIRECTORIO_MODELOS = Path(os.environ.get('SICDA_MODELOS', MODELOS))
VERSIONES = {'complement_nb': None, 'linear_svc': None}

categorias = json.loads((DATOS_PROCESADOS / 'categorias.json').read_text(encoding='utf-8'))
nombre_dominio = {d['id']: d['codigo'] for d in categorias['dominios']}
nombre_area = {a['id']: a['nombre'] for d in categorias['dominios'] for a in d['areas']}
area_a_dominio = {a['id']: d['id'] for d in categorias['dominios'] for a in d['areas']}

metadatos = {alg: cargar_metadatos(alg, v, DIRECTORIO_MODELOS) for alg, v in VERSIONES.items()}
pd.DataFrame({alg: {'version': m['version'], **m['datos']} for alg, m in metadatos.items()})

## Comparación en validación

El criterio de selección es el **F1-macro jerárquico** (dominio y área correctos). Como los
clasificadores siempre eligen un área del dominio predicho, coincide con el F1-macro de área.

In [ ]:
filas = []
for alg, m in metadatos.items():
    for tipo, info in m['tipos'].items():
        met = info['metricas_validacion']
        filas.append({
            'modelo': f'{alg} · {tipo}',
            'f1_macro_dominio': met['dominio']['f1_macro'],
            'f1_macro_area': met['area']['f1_macro'],
            'f1_macro_jerarquico': met['jerarquico']['f1_macro'],
            'accuracy_jerarquica': met['jerarquico']['accuracy'],
            'f1_ponderado_area': met['area']['f1_ponderado'],
            'revision_%': 100 * info['porcentaje_revision_validacion'],
            'ms_por_documento': info['tiempo_prediccion']['media_ms'],
            'entrenamiento_s': info['tiempo_entrenamiento_s'],
        })
comparacion = pd.DataFrame(filas).set_index('modelo').sort_values('f1_macro_jerarquico', ascending=False)
comparacion.round(3)

In [ ]:
# Metas RNF-02 (F1-macro ≥ 0,85 en dominio y ≥ 0,65 en área) y RNF-01 (< 2 s por resumen).
metas = pd.DataFrame({
    'dominio ≥ 0,85': comparacion['f1_macro_dominio'] >= 0.85,
    'área ≥ 0,65': comparacion['f1_macro_area'] >= 0.65,
    '< 2 s por resumen': comparacion['ms_por_documento'] < 2000,
})
metas

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
comparacion[['f1_macro_dominio', 'f1_macro_area']].plot.barh(ax=ax)
ax.axvline(0.85, ls='--', c='C0', lw=1)
ax.axvline(0.65, ls='--', c='C1', lw=1)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel('F1-macro en validación (líneas: metas RNF-02)')
ax.set_title('Modelos clásicos')
plt.tight_layout()

## Hiperparámetros elegidos

Cada submodelo tiene su propia búsqueda (`GridSearchCV`, 5 particiones estratificadas sobre
train, F1-macro). `terminos` es el tamaño del vocabulario TF-IDF del submodelo.

In [ ]:
hiperparametros = pd.DataFrame([
    {
        'algoritmo': alg,
        'tipo': tipo,
        'submodelo': sub,
        **{k.removeprefix('clf__'): v for k, v in r['mejores_parametros'].items()},
        'f1_macro_cv': r['f1_macro_cv'],
        'clases': r['clases'],
        'terminos': r['terminos'],
    }
    for alg, m in metadatos.items()
    for tipo, info in m['tipos'].items()
    for sub, r in info['submodelos'].items()
])
hiperparametros.round(4)

In [ ]:
fig, ejes = plt.subplots(1, len(metadatos), figsize=(11, 3.5))
for ax, (alg, m) in zip(ejes, metadatos.items(), strict=True):
    for tipo, info in m['tipos'].items():
        submodelo = 'dominio' if tipo == 'jerarquico' else 'plano'
        cv = pd.DataFrame(info['submodelos'][submodelo]['resultados_cv'])
        if cv.empty:
            continue
        parametro = next(iter(cv['parametros'][0]))
        x = cv['parametros'].map(lambda p, k=parametro: p[k])
        ax.errorbar(x, cv['media'], yerr=cv['desviacion'], marker='o', capsize=3,
                    label=f'{tipo} ({submodelo})')
        ax.set_xscale('log')
        ax.set_xlabel(parametro.removeprefix('clf__'))
    ax.set_title(alg)
    ax.set_ylabel('F1-macro (CV)')
    ax.legend()
plt.tight_layout()

## Análisis de errores del mejor modelo clásico

In [ ]:
MEJOR = comparacion.index[0]
algoritmo, tipo = MEJOR.split(' · ')
version = metadatos[algoritmo]['version']
predicciones = cargar_predicciones(algoritmo, tipo, version, DIRECTORIO_MODELOS)
print(f'Mejor modelo clásico: {MEJOR} (versión {version})')

# Áreas con peor F1 en validación.
reporte = reporte_por_clase(predicciones['area_real'], predicciones['area_pred'], nombre_area)
reporte.insert(1, 'dominio', [nombre_dominio[area_a_dominio[a]] for a in reporte.index])
reporte.head(15).round(3)

In [ ]:
matriz = matriz_confusion(predicciones['dominio_real'], predicciones['dominio_pred'],
                          sorted(nombre_dominio), nombre_dominio)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(matriz.div(matriz.sum(axis=1), axis=0), annot=True, fmt='.2f', cmap='Blues', ax=ax)
ax.set_title(f'Dominio · {MEJOR} (proporción por fila)')
plt.tight_layout()

In [ ]:
# Pares de áreas que más se confunden (real → predicha).
errores = predicciones[predicciones['area_real'] != predicciones['area_pred']]
confusiones = (errores.groupby(['area_real', 'area_pred']).size().rename('documentos')
               .sort_values(ascending=False).head(15).reset_index())
confusiones['real'] = confusiones['area_real'].map(nombre_area)
confusiones['predicha'] = confusiones['area_pred'].map(nombre_area)
confusiones[['real', 'predicha', 'documentos']]

## Ejemplo de clasificación con explicación

El modelo publicado incluye el `Preprocesador`, así que recibe el resumen crudo, igual que la API.

In [ ]:
modelo = cargar_modelo(algoritmo, tipo, version, DIRECTORIO_MODELOS)
val = pd.read_parquet(DATOS_PROCESADOS / 'val.parquet')
documento = val.iloc[0]
resultado = modelo.predecir_detallado([documento['texto']])[0]

print(documento['texto'][:400], '…\n')
print(f"Real:     {documento['dominio']} / {documento['area']}")
print(f"Predicho: {nombre_dominio[resultado['dominio']['id']]} ({resultado['dominio']['confianza']:.2f})"
      f" / {nombre_area[resultado['area']['id']]} ({resultado['area']['confianza']:.2f})")
print(f"Confianza conjunta {resultado['confianza_conjunta']:.2f} · requiere revisión: {resultado['requiere_revision']}")
print('Alternativas de área:', [(nombre_area[a['id']], round(a['confianza'], 2)) for a in resultado['area']['alternativas']])
print('Términos clave:', [t['termino'] for t in resultado['terminos_clave']])

La comparación final con DistilBERT, la calibración de los umbrales y la única evaluación en
test están en el notebook 05.